# Evaluate Ensemble

B Steele, Colorado State University, ROSSyndicate — last updated Sep 15, 2026

This notebook assembles the production 5-seed × 2-model ensemble trained in `04_make_models.ipynb`, evaluates it on the fixed holdout built in `03_split_data.ipynb`, and validates its feature attributions with SHAP.

This replaces the original `05_split_seed_sensitivity.Rmd`, which asked whether a single train/test split's random seed materially changes reported performance. That question is now answered by construction: the production model already averages over 5 independent HUC8-random CV arrangements rather than trusting any one of them, so this notebook's job is to report on the resulting ensemble and check that it behaves sensibly — where its error concentrates, and whether the features it actually relies on line up with physical expectation — rather than to re-litigate split sensitivity.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import xgboost as xgb

sys.path.insert(0, str(Path("python").resolve()))
import features
import metrics as M
import model_lgb
import model_xgb
import spatial_cv

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "v3_production"
TARGET = "harmonized_value"
MODELS = {"xgboost": model_xgb, "lightgbm": model_lgb}
ENSEMBLE_SEEDS = spatial_cv.ENSEMBLE_SEEDS

ROSS_LT_PAL = ["#002EA3", "#E70870", "#256BF5", "#745CFB", "#1E4D2B", "#56104E"]

df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
holdout = df[df["is_holdout"]].reset_index(drop=True)
print(f"holdout: {len(holdout):,} rows")

## Assemble the ensemble

Each seed independently selected its own features and hyperparameters and produced holdout predictions from its own 4 fold models, per algorithm. The production prediction for a given (site, date) is the plain average across all 5 seeds × 2 algorithms × 4 folds — 40 models — with no per-seed or per-model weighting.

In [ ]:
all_rows = []
for seed in ENSEMBLE_SEEDS:
    all_rows.append(pd.read_parquet(MODELS_DIR / f"seed{seed}" / "holdout_predictions.parquet"))
all_preds = pd.concat(all_rows, ignore_index=True)

ensemble_pred = all_preds.groupby(["siteSR_id", "date"]).agg(
    y=("y", "first"), pred=("pred", "mean"), HUC4=("HUC4", "first"), HUC8=("HUC8", "first")).reset_index()
ensemble_metrics = M.all_metrics(ensemble_pred["y"], ensemble_pred["pred"])
print(f"v3 production ensemble (5 seeds x 2 models = 40 models): {ensemble_metrics}")

per_model_metrics = {}
for m in MODELS:
    sub = all_preds[all_preds["model"] == m].groupby(["siteSR_id", "date"]).agg(
        y=("y", "first"), pred=("pred", "mean")).reset_index()
    per_model_metrics[m] = M.all_metrics(sub["y"], sub["pred"])
    print(f"{m}-only ensemble (5 seeds): {per_model_metrics[m]}")

## Predicted vs. observed

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ensemble_pred["y"], ensemble_pred["pred"], alpha=0.3, s=12, color=ROSS_LT_PAL[0])
lims = [0, max(ensemble_pred["y"].max(), ensemble_pred["pred"].max())]
ax.plot(lims, lims, ls="--", color="grey")
ax.set_xlabel("observed Secchi depth (m)")
ax.set_ylabel("predicted Secchi depth (m)")
ax.set_title(f"production ensemble, fixed holdout (n={len(ensemble_pred):,})\n"
             f"RMSE={ensemble_metrics['rmse']:.3f}m  MAE={ensemble_metrics['mae']:.3f}m  "
             f"bias={ensemble_metrics['bias']:+.3f}m  R\u00b2={ensemble_metrics['r2']:.3f}")
plt.tight_layout()
plt.show()

## Error vs. match-up time gap

Each holdout point pairs an in-situ Secchi sample with the nearest available satellite overpass within a 5-day window (`02_make_matches.Rmd` retains only the closest-in-time observation per site/date group). `time_diff` records that signed offset and is deliberately excluded from the feature set (`features.NON_FEATURE_COLS`) since it doesn't exist for an arbitrary pass at deployment time. This checks whether holdout error still tracks the size of that offset — i.e., whether looser match-ups are a quiet source of degraded predictions.

In [ ]:
gap_lookup = df[["siteSR_id", "date", "time_diff"]].drop_duplicates(subset=["siteSR_id", "date"])
gap_pred = ensemble_pred.merge(gap_lookup, on=["siteSR_id", "date"], how="left")
gap_pred["gap_days"] = gap_pred["time_diff"].abs()
gap_pred["abs_error"] = (gap_pred["pred"] - gap_pred["y"]).abs()

GAP_BIN_WIDTH = 0.5
MIN_BIN_N = 10
gap_pred["gap_bin"] = (gap_pred["gap_days"] // GAP_BIN_WIDTH) * GAP_BIN_WIDTH
binned = gap_pred.groupby("gap_bin")["abs_error"].agg(["mean", "sem", "count"]).reset_index()
binned = binned[binned["count"] >= MIN_BIN_N]

gap_corr = gap_pred["gap_days"].corr(gap_pred["abs_error"])
print(f"Pearson r(gap_days, abs_error) = {gap_corr:.3f}  (n={len(gap_pred):,})")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(gap_pred["gap_days"], gap_pred["abs_error"], alpha=0.25, s=10, color=ROSS_LT_PAL[0],
           label="holdout obs.")
ax.errorbar(binned["gap_bin"] + GAP_BIN_WIDTH / 2, binned["mean"], yerr=binned["sem"],
            color=ROSS_LT_PAL[1], marker="o", ms=5, lw=2, capsize=3,
            label=f"binned mean ±SEM ({GAP_BIN_WIDTH:g}-day bins, n≥{MIN_BIN_N})")
ax.set_xlabel("time gap between in-situ sample and satellite overpass (days)")
ax.set_ylabel("absolute prediction error (m)")
ax.set_title(f"holdout error vs. match-up time gap (n={len(gap_pred):,})\nPearson r={gap_corr:.3f}")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Performance by HUC8

Error is not uniform across basins. The CV/holdout split itself is built at the HUC8 level (`03_split_data.ipynb`), so basin-level performance is reported at that same granularity here rather than aggregated up to HUC4 — a HUC4 can straddle both the holdout and multiple CV folds, but every HUC8 sits entirely on one side. There is no HUC8 boundary-polygon layer on hand, so rather than a choropleth, each holdout site is plotted individually, colored by its own HUC8 group's error, with HUC4 boundaries drawn only as spatial context. Basin-level difficulty in this dataset has consistently traced back to physical causes (training-envelope extrapolation, genuinely low optical signal) documented in the earlier outlier-HUC investigation, not to data quality problems that would call for dropping anything.

In [ ]:
huc8_metrics = (ensemble_pred.groupby("HUC8")
                .apply(lambda g: pd.Series({**M.all_metrics(g["y"], g["pred"]), "HUC4": g["HUC4"].iloc[0]}),
                       include_groups=False)
                .reset_index()
                .sort_values("rmse", ascending=False))
huc8_metrics

In [ ]:
states = gpd.read_file(AQUAMATCH_DIR / "states.gdb", layer="states")
huc4s = gpd.read_file(AQUAMATCH_DIR / "huc4_filtered.gdb", layer="huc4_filtered")

latlon_lookup = df[["siteSR_id", "date", "lat", "lon"]].drop_duplicates(subset=["siteSR_id", "date"])
site_metrics = (ensemble_pred.merge(latlon_lookup, on=["siteSR_id", "date"], how="left")
                .merge(huc8_metrics[["HUC8", "rmse", "bias"]], on="HUC8", how="left"))
site_gdf = gpd.GeoDataFrame(site_metrics, geometry=gpd.points_from_xy(site_metrics.lon, site_metrics.lat),
                            crs="EPSG:4326").to_crs(huc4s.crs)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
states.plot(ax=ax, color="whitesmoke", edgecolor="grey")
huc4s.boundary.plot(ax=ax, color="grey", linewidth=0.4)
site_gdf.plot(ax=ax, column="rmse", cmap="viridis", legend=True, markersize=10, alpha=0.8,
              legend_kwds={"label": "holdout RMSE (m), by site's HUC8"})
ax.set_title("holdout RMSE by HUC8 (site-level)")
ax.set_axis_off()

ax = axes[1]
states.plot(ax=ax, color="whitesmoke", edgecolor="grey")
huc4s.boundary.plot(ax=ax, color="grey", linewidth=0.4)
vmax = site_gdf["bias"].abs().max()
site_gdf.plot(ax=ax, column="bias", cmap="RdBu_r", vmin=-vmax, vmax=vmax, legend=True, markersize=10, alpha=0.8,
              legend_kwds={"label": "holdout bias (m), by site's HUC8; + = overprediction"})
ax.set_title("holdout bias by HUC8 (site-level)")
ax.set_axis_off()

fig.tight_layout()
plt.show()

### Predicted vs. observed in the worst-performing HUC8s

Small multiples for the worst-RMSE HUC8s (restricted to `n >= 15` so this isn't dominated by tiny, noisy groups), each panel on its own 1:1 line.

In [ ]:
N_WORST = 6
MIN_N = 15
worst_huc8 = huc8_metrics[huc8_metrics["n"] >= MIN_N].sort_values("rmse", ascending=False).head(N_WORST)

n_panels = len(worst_huc8)
ncols = 3
nrows = -(-n_panels // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)

for ax, (_, row) in zip(axes.flat, worst_huc8.iterrows()):
    sub = ensemble_pred[ensemble_pred["HUC8"] == row["HUC8"]]
    ax.scatter(sub["y"], sub["pred"], alpha=0.6, s=25, color=ROSS_LT_PAL[0])
    lims = [0, max(sub["y"].max(), sub["pred"].max())]
    ax.plot(lims, lims, ls="--", color="grey")
    ax.set_title(f"HUC8 {row['HUC8']} (HUC4 {row['HUC4']})\n"
                 f"n={int(row['n'])}, RMSE={row['rmse']:.2f}m, bias={row['bias']:+.2f}m")
    ax.set_xlabel("observed Secchi (m)")
    ax.set_ylabel("predicted Secchi (m)")

for ax in axes.flat[n_panels:]:
    ax.set_visible(False)

fig.suptitle(f"predicted vs. observed, {N_WORST} worst-RMSE HUC8s (n >= {MIN_N})", y=1.02)
fig.tight_layout()
plt.show()

## SHAP feature attribution

Each seed selects a slightly different final feature set (18–25 features typically). For a clean cross-seed comparison, SHAP is computed for every (seed, model) pair on the shared holdout, restricted to the **unanimous feature core** — the intersection of all 10 (seed, model) feature sets — and averaged into one ensemble-mean attribution per feature. Models are retrained here from each seed's saved feature set and gap-aware-tuned hyperparameters (`04_make_models.ipynb`'s `backward_elim_summary.json` / `final_tune_{model}.json`), not re-selected or re-tuned.

In [ ]:
def load_seed_config(seed, model_name):
    with open(MODELS_DIR / f"seed{seed}" / "backward_elim_summary.json") as fh:
        feats = json.load(fh)["intersection_of_per_model"]
    with open(MODELS_DIR / f"seed{seed}" / f"final_tune_{model_name}.json") as fh:
        params = json.load(fh)["best_params"]
    return feats, params


def shap_for_model(model_name, model, holdout, feats):
    explainer = shap.TreeExplainer(model)
    if model_name == "xgboost":
        sv = explainer.shap_values(xgb.DMatrix(holdout[feats]))
    else:
        sv = explainer.shap_values(holdout[feats])
    return np.asarray(sv)


per_seed_model = {}
for seed in ENSEMBLE_SEEDS:
    folds = spatial_cv.build_folds(df, seed)
    for model_name, mod in MODELS.items():
        feats, params = load_seed_config(seed, model_name)
        print(f"seed{seed}/{model_name}: training 4 folds + SHAP on {len(holdout)}-row holdout ({len(feats)} feats)")
        fold_shaps = []
        for f in folds:
            model = mod.train_fold_models([f], feats, TARGET, params)[0]
            fold_shaps.append(shap_for_model(model_name, model, holdout, feats))
        mean_shap = np.mean(fold_shaps, axis=0)
        per_seed_model[(seed, model_name)] = pd.DataFrame(mean_shap, columns=feats)

unanimous = None
for feats_df in per_seed_model.values():
    cols = set(feats_df.columns)
    unanimous = cols if unanimous is None else (unanimous & cols)
unanimous = sorted(unanimous)
print(f"\nunanimous core ({len(unanimous)} features): {unanimous}")

In [ ]:
long_rows = []
for (seed, model_name), feats_df in per_seed_model.items():
    sub = feats_df[unanimous].copy()
    sub["seed"] = seed
    sub["model"] = model_name
    sub["_row"] = np.arange(len(holdout))
    long_rows.append(sub)
long_df = pd.concat(long_rows, ignore_index=True)

ensemble_mean_shap = long_df.groupby("_row")[unanimous].mean().sort_index()

SITE_COLS = {"elevation_m", "catchment_area_sqkm", "pct_impervious_2006", "pct_urban_2006",
             "pct_forest_2006", "pct_cropland_2006", "pct_wetland_2006", "shore_flag"}
WEATHER_PREFIXES = ("precip_mm_prev", "tmax_degC_prev", "tmean_degC_prev", "tmin_degC_prev", "srad_Wm2_prev")


def categorize(feat):
    if feat in SITE_COLS:
        return "site"
    if feat.startswith(WEATHER_PREFIXES):
        return "weather"
    return "optical"


mean_abs_shap = ensemble_mean_shap.abs().mean().sort_values(ascending=False)
cat_share = mean_abs_shap.groupby(categorize).sum() / mean_abs_shap.sum() * 100
print("top 8 features by mean|SHAP|:")
print(mean_abs_shap.head(8).round(4))
print("\nshare of total mean|SHAP| attribution by category:")
print(cat_share.round(1))

fig, ax = plt.subplots(figsize=(8, 6))
colors = mean_abs_shap.index.map(lambda f: {"optical": ROSS_LT_PAL[0], "site": ROSS_LT_PAL[1],
                                             "weather": ROSS_LT_PAL[2]}[categorize(f)])
ax.barh(mean_abs_shap.index[::-1], mean_abs_shap.values[::-1], color=colors[::-1])
ax.set_xlabel("mean |SHAP| (m)")
ax.set_title("ensemble-mean feature attribution (unanimous core)")
plt.tight_layout()
plt.show()

## Per-HUC8 SHAP: does the model lean on different features in different basins?

In [ ]:
shap_with_huc8 = ensemble_mean_shap.copy()
shap_with_huc8["HUC8"] = holdout["HUC8"].values
huc8_shap = shap_with_huc8.groupby("HUC8")[unanimous].apply(lambda g: g.abs().mean())

if "catchment_area_sqkm" in huc8_shap.columns:
    print("catchment_area_sqkm mean|SHAP| by HUC8 (top 5):")
    print(huc8_shap["catchment_area_sqkm"].sort_values(ascending=False).head(5).round(4))

fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(huc8_shap))))
im = ax.imshow(huc8_shap.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(unanimous)))
ax.set_xticklabels(unanimous, rotation=90)
ax.set_yticks(range(len(huc8_shap)))
ax.set_yticklabels(huc8_shap.index)
ax.set_title("mean |SHAP| by HUC8 and feature (unanimous core)")
fig.colorbar(im, ax=ax, label="mean |SHAP| (m)")
plt.tight_layout()
plt.show()

## Production summary

The production model is a 40-member ensemble — 5 independent HUC8-random CV arrangements × 2 algorithms (XGBoost, LightGBM) × 4 folds each — trained unweighted, with gap-aware hyperparameter selection at every tuning stage and per-seed feature sets chosen by correlation pruning followed by shadow-decoy backward elimination (run per-model and jointly across both algorithms). Every member is evaluated on one fixed HUC8-random holdout that no model ever trains on. Artifacts (fold models, per-seed feature/hyperparameter configs, holdout predictions) are saved under `xg_models/v3_production/seed{501–505}/`.

**Feature set.** Per-seed final sets range 23–26 features; **20 features are unanimous** across all 5 seeds × 2 algorithms: `BG`, `BR`, `MNDWI`, `NDSSI`, `NDVI`, `NR`, `atm_corr_LaSRC`, `catchment_area_sqkm`, `fai`, `green_corr7`, `nir_corr7`, `pct_cropland_2006`, `pct_forest_2006`, `pct_impervious_2006`, `pct_wetland_2006`, `srad_Wm2_prev1`, `srad_Wm2_prev30`, `temp_corr7`, `tmax_degC_prev30`, `tmean_degC_prev30`. By SHAP attribution, optical/spectral features carry 76.7% of total mean|SHAP|, site characteristics 17.2%, antecedent weather 6.1% — non-remote-sensing variables are a minority of the feature count but a material share of what actually drives predictions. The top individual features (`BG`, `green_corr7`, `BR`, `fai`) are exactly the literature-cited clarity/chlorophyll band ratios this project's feature engineering was built around.

**Holdout performance.** RMSE 1.271 m, MAE 0.979 m, R² 0.610, over 1,851 unique (site, date) holdout observations. XGBoost-only and LightGBM-only sub-ensembles are nearly identical (RMSE 1.270 m vs. 1.274 m) — the two algorithms agree closely, consistent with algorithm/seed ensembling buying prediction stability more than large point-estimate correction. The ensemble carries a **+0.400 m systematic overprediction bias (11.3%)** — a real, current characteristic of this configuration worth further investigation (e.g., against the SDD-weighted alternative in `python/weighting.py`, or a bias-correction step), not one this analysis resolves. Holdout error is unrelated to the in-situ/overpass match-up time gap (Pearson r=0.019 between absolute error and offset magnitude across the full 0–5 day matching window) — the closest-available-pass selection in `02_make_matches.Rmd` is not itself a source of degraded predictions.

**Where error concentrates, and why.** Basin-level error is reported at the HUC8 level (the actual split granularity) rather than HUC4. The two largest HUC8 units inside HUC4 1701 remain the hardest large-sample basins (HUC 17010203, n=508, RMSE 1.33 m; HUC 17010101, n=70, RMSE 1.63 m) — consistent with the earlier outlier-HUC investigation's finding that this forested lake cluster has a genuinely weak optical clarity signal, not a data-quality problem. HUC 14070005 (Lake Powell, HUC4 1407) has an elevated RMSE (1.37 m) but a strong R² (0.83, n=25) and underprediction bias (−0.81 m); SHAP shows `catchment_area_sqkm` is the single most-relied-on feature there (3rd-highest mean|SHAP| of all 28 holdout HUC8s) — independently reproducing, from this fresh dataset and split, the original training-envelope finding that Lake Powell's catchment area sits far outside the bulk of the training distribution. HUC 10040104 (HUC4 1004, n=133, RMSE 1.82 m) is now the single worst large-sample basin and was not previously flagged — worth a closer look in a future pass, though nothing in this analysis points to a data problem there. As with the earlier outlier-HUC and partition-sensitivity work, none of this is a case for dropping data: the paths forward are more splits/ensembling or basin-aware feature work, not exclusion.

**Context on the numbers.** These holdout metrics are not directly comparable to the earlier v3 research pipeline's published numbers (RMSE 1.39 m, bias +0.01 m) — the upstream `01`/`02` filtering has since been reworked (a revised multiband RANSAC filter, a smaller site set: 1,462 vs. 1,525 sites), so this is evaluated on a materially different dataset and a freshly-drawn HUC8-random split, not a regression against that prior result.